In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dataclasses as dcls

In [ ]:
@dcls.dataclass(frozen=True)
class GWConfig:
	# constants
	M1 = 0.1
	M2 = 0.1
	R = 1
	G = 1
	c = 10.0          # artificially small speed of light for demonstration
	mu = (M1 * M2) / (M1 + M2)

	# numerics
	steps = 2000
	timestep = 0.01

	@property
	def GMM(self):
		return self.G * self.M1 * self.M2


cfg = GWConfig()

In [ ]:


pos1 = np.array([-0.5,0], dtype=np.float64)
pos2 = pos1 - cfg.R * pos1/np.linalg.norm(pos1)

v1 = np.array([0, 0.25], dtype=np.float64)
v2 = np.array([0,-0.25], dtype=np.float64)
# f = Gm1m2/r^2

E = 0.5 * cfg.M1 * (v1**2) + 0.5 * cfg.M2 * (v2**2) - cfg.GMM / np.abs(np.linalg.norm(pos2 - pos1))
print(E)
state = np.concatenate((pos1, pos2, v1, v2))


def a_2p5PN_rr(rvec, vvec, G, M, mu, c=1.0):
	r = np.linalg.norm(rvec)
	v2 = np.dot(vvec, vvec)
	rv = np.dot(rvec, vvec)
	factor = - (8.0 / 5.0) * (G**3 * M * mu) / (c**5 * r**3)
	
	bracket_term = (v2 + 3.0*G*M/r) * rvec - 3.0*rv * vvec
	return factor * bracket_term


def f(state):
	x1, y1, x2, y2, vx1, vy1, vx2, vy2 = state
	pos1 = np.array([x1, y1])
	pos2 = np.array([x2, y2])
	v1   = np.array([vx1, vy1])
	v2   = np.array([vx2, vy2])
	rvec = pos2 - pos1
	r = np.linalg.norm(rvec)

	a1 = (cfg.G * cfg.M2 / r**3) * rvec
	a2 = (cfg.G * cfg.M1 / r**3) * (-rvec)

	a_rr = a_2p5PN_rr(rvec, v2-v1)

	dx1dt, dy1dt = v1
	dx2dt, dy2dt = v2
	dvx1dt, dvy1dt = a1
	dvx2dt, dvy2dt = a2
	
	return np.array([
			dx1dt, dy1dt,
			dx2dt, dy2dt,
			dvx1dt, dvy1dt,
			dvx2dt, dvy2dt
	], dtype=float)

def rk4_step(state, dt):
	k1 = f(state)
	k2 = f(state + 0.5*dt*k1)
	k3 = f(state + 0.5*dt*k2)
	k4 = f(state + dt*k3)
	
	new_state = state + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
	return new_state


locs = []
for step in range(cfg.steps):
	locs.append(state[:4])

	state = rk4_step(state, cfg.timestep)

	x1, y1, x2, y2, vx1, vy1, vx2, vy2 = state
	rvec = np.array([x2 - x1, y2 - y1])
	r = np.linalg.norm(rvec)
	kinetic = 0.5 * cfg.M1 * (vx1**2 + vy1**2) + 0.5 * cfg.M2 * (vx2**2 + vy2**2)
	potential = - cfg.GMM / r
	E = kinetic + potential
	if step % 200 == 0:
			print(f"step {step}: E = {E:.6f}, r = {r:.6f}")

locs = np.array(locs)
print(locs.shape)
fig, ax = plt.subplots(figsize=(15, 10))
plt.plot(locs[:,0], locs[:,1], label="Mass 1")
plt.plot(locs[:,2], locs[:,3], label="Mass 2")
ax.scatter([locs[-1,0], locs[-1,2]], [locs[-1,1], locs[-1,3]], 
            color=['blue','orange'], zorder=3)
ax.set(title="Two-Body Orbit with RK4", xlabel="x", ylabel="y", )
plt.legend()
plt.gca().set_aspect("equal", "box")
plt.show()

# for step in range(steps):
#     # updates position based on velocity
#     pos1 += v1 * timestep 
#     pos2 += v2 * timestep

#     # Calculates acceleration between the two masses
#     rvec = pos2 - pos1
#     a1 = G * M2 / (np.linalg.norm(rvec)**2) * rvec
#     a2 = G * M1 / (np.linalg.norm(rvec)**2) * -rvec

#     # Updates velocity based on the force
#     v1 += a1 * timestep
#     v2 += a2 * timestep
#     E = 0.5*M1*(np.linalg.norm(v1)**2)+0.5*M2*(np.linalg.norm(v2)**2)-G*M1*M2/np.abs(np.linalg.norm(rvec))
#     print(E)
#     if step % 10 == 0:
#         holes = np.stack([pos1,pos2], axis=1)
#         plt.scatter(holes[0], holes[1])
#         dirs = np.stack([v1, v2], axis=1)
#         plt.quiver(holes[0], holes[1], dirs[0], dirs[1])

#         plt.xlim(-1,1)
#         plt.ylim(-1,1)
#         # plt.show()


# grid_size = 50
# X, Y = np.meshgrid(*[np.linspace(-np.pi, np.pi, grid_size) for _ in range(2)])

# fig = plt.figure()
# ax = fig.add_subplot(111)
# ax.set_xlim(-np.pi, np.pi)
# ax.set_ylim(-np.pi, np.pi)
# ax.set_title("GW Wave Test")
# holes = np.stack([pos1,pos2], axis=1)
# ax.scatter(holes[0], holes[1])
# plt.show()


In [ ]:
import numpy as np

def a_newtonian(rvec, M, r):
	"""
	Newtonian (relative) acceleration for total mass M = M1 + M2
	(assuming reduced mass mu is in orbit).
	"""
	return -cfg.G * M / r**3 * rvec

def a_2p5PN_rr(rvec, vvec,M, mu):
	"""
	Leading-order GW radiation-reaction acceleration (2.5PN) 
	for a binary in the center of mass frame.
	rvec = relative position (r1 - r2)
	vvec = relative velocity (v1 - v2)
	M    = M1 + M2
	mu   = (M1*M2)/(M1+M2)
	c    = speed of light
	"""
	r = np.linalg.norm(rvec)
	v2 = np.dot(vvec, vvec)
	rv = np.dot(rvec, vvec)
	factor = - (8.0 / 5.0) * (cfg.G**3 * M * mu) / (cfg.c**5 * r**3)
	
	bracket_term = (v2 + 3.0*cfg.G*M/r) * rvec - 3.0*rv * vvec
	return factor * bracket_term

# Example integrator (schematic):
def rk4_step_rel(rvec, vvec, dt, M, mu):
	"""
	Evolve the relative coordinate (rvec, vvec) one RK4 step 
	including both Newtonian + 2.5PN terms.
	"""
	def deriv(r, v):
			# Newtonian acceleration
			r_mag = np.linalg.norm(r)
			a_newt = a_newtonian(r, M, r_mag)
			# 2.5PN RR acceleration
			a_rr   = a_2p5PN_rr(r, v, M, mu)
			return np.concatenate([v, a_newt + a_rr])
	
	y0 = np.concatenate([rvec, vvec])
	
	k1 = deriv(rvec, vvec)
	k2 = deriv(rvec + 0.5*dt*k1[0:2], vvec + 0.5*dt*k1[2:4])
	k3 = deriv(rvec + 0.5*dt*k2[0:2], vvec + 0.5*dt*k2[2:4])
	k4 = deriv(rvec + dt*k3[0:2],     vvec + dt*k3[2:4])
	
	y_new = y0 + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
	r_new = y_new[0:2]
	v_new = y_new[2:4]
	return r_new, v_new


M  = cfg.M1 + cfg.M2        # total mass
mu = (cfg.M1*cfg.M2) / M    # reduced mass
dt = cfg.timestep

# Relative initial conditions (place COM at origin or similar)
rvec = np.array([1.0, 0.0])
vvec = np.array([0.0, 0.3])  # tangential velocity

for step in range(cfg.steps):
    rvec, vvec = rk4_step_rel(rvec, vvec, dt, M, mu)
    
    # Every so often, check separation
    if step % 1000 == 0:
        r = np.linalg.norm(rvec)
        print(f"step={step}, r={r:.6f}")
    
    # Stop if they merge
    if np.linalg.norm(rvec) < 0.01:
        print("Merged!")
        break


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

M  = cfg.M1 + cfg.M2        # total mass
mu = (cfg.M1*cfg.M2) / M    # reduced mass

num_steps = 30000

# ------------------------------------------------
# 2.5PN Radiation-Reaction Accelerations
# ------------------------------------------------
def a_newtonian(rvec, M):
	"""
	Newtonian (relative) acceleration between two masses in COM frame:
	a_newt = - G M / r^3 * rvec
	"""
	r = np.linalg.norm(rvec)
	return -cfg.G*M / r**3 * rvec

def a_2p5PN_rr(rvec, vvec, M, mu):
	"""
	Leading-order 2.5PN radiation-reaction acceleration (relative) 
	for two non-spinning bodies.
	
		a_RR = - (8/5) * (G^3 M mu / (c^5 r^3)) *
							[ (v^2 + 3 G M / r) * rvec - 3 (rvec · vvec) vvec ]
	"""
	r = np.linalg.norm(rvec)
	v2 = np.dot(vvec, vvec)
	rv = np.dot(rvec, vvec)
	
	factor = - (8.0/5.0) * (cfg.G**3 * M * mu) / (cfg.c**5 * r**3)
	bracket = (v2 + 3.0*cfg.G*M/r)*rvec - 3.0*rv*vvec
	
	return factor * bracket

# ------------------------------------------------
# A simple RK4 step in the relative frame
# ------------------------------------------------
def rk4_step_rel(rvec, vvec, dt, M, mu):
    """
    Evolve the relative coordinate (rvec, vvec) one RK4 step 
    with Newtonian + 2.5PN terms.
    
    y = [r_x, r_y, v_x, v_y]
    dy/dt = [v_x, v_y, a_x, a_y]
    """
    def deriv(r, v):
        a_newt = a_newtonian(r, M)
        a_rr   = a_2p5PN_rr(r, v, M, mu)
        return np.concatenate([v, a_newt + a_rr])
    
    y0 = np.concatenate([rvec, vvec])
    
    k1 = deriv(rvec,                  vvec)
    k2 = deriv(rvec + 0.5*dt*k1[0:2], vvec + 0.5*dt*k1[2:4])
    k3 = deriv(rvec + 0.5*dt*k2[0:2], vvec + 0.5*dt*k2[2:4])
    k4 = deriv(rvec + dt*k3[0:2],     vvec + dt*k3[2:4])
    
    y_new = y0 + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
    r_new = y_new[0:2]
    v_new = y_new[2:4]
    return r_new, v_new

# ------------------------------------------------
# Initial Conditions (relative frame)
# ------------------------------------------------
# Start them on a roughly circular orbit:
rvec = np.array([1.0, 0.0])   # separation vector
vvec = np.array([0.0, 0.3])   # relative velocity

# Lists to store positions for plotting
r1_list = []
r2_list = []

for step in range(cfg.steps):
    # Convert from relative coords to each mass’s position
    # Center-of-mass is at origin (if total momentum = 0).
    pos1 = (cfg.M2 / M)*rvec    # r1 = + (m2/M)*r
    pos2 = -(cfg.M1 / M)*rvec   # r2 = - (m1/M)*r
    
    r1_list.append(pos1.copy())
    r2_list.append(pos2.copy())
    
    # Integrate one step with RK4
    rvec, vvec = rk4_step_rel(rvec, vvec, dt, M, mu)
    
    # Check if they've merged or gotten really close
    if np.linalg.norm(rvec) < 0.005:
        print(f"Objects have merged at step {step}")
        break

# Convert lists to arrays for plotting
r1_array = np.array(r1_list)  # shape (steps, 2)
r2_array = np.array(r2_list)  # shape (steps, 2)

# ------------------------------------------------
# Plot
# ------------------------------------------------
plt.figure(figsize=(6,6))
plt.plot(r1_array[:,0], r1_array[:,1], label="Mass 1")
plt.plot(r2_array[:,0], r2_array[:,1], label="Mass 2")

# Mark final positions
plt.scatter(r1_array[-1,0], r1_array[-1,1], color="blue", s=50)
plt.scatter(r2_array[-1,0], r2_array[-1,1], color="orange", s=50)

plt.xlabel("x")
plt.ylabel("y")
plt.title("2-Body Inspiral with 2.5PN GW Backreaction")
plt.legend()
plt.axis("equal")
plt.tight_layout()
plt.show()
